<h1 align="center">Limpieza y preparación de datos</h1>

## U3.01 – Detección e Identificación de Problemas en los Datos

### Cargar e inspeccionar datos iniciales

In [31]:
import pandas as pd
import numpy as np

# Crear un DataFrame “sucio” de ejemplo
df = pd.DataFrame({
    'id_cliente': [1, 2, 2, 3, 4, 5, 6, 6, None],
    'nombre': [' Ana  ', 'CARLOS', 'CARLOS', 'María', None, 'Juan', 'Lucia', 'Lucia', 'Ana'],
    'edad': [25, 40, 40, None, 17, 80, 35, 35, 25],
    'ingresos_mensuales': [1500.0, 2500.50, 2500.50, 3200.0, 500.0, None, 1800.0, 1800.0, 1500.0],
    'ciudad': ['Madrid', 'Barcelona', 'Barcelona', 'Madrid', 'MADRID', 'Valencia', 'Barcelona', 'Barcelona', 'Madrid'],
    'segmento': ['Silv', 'Gol', 'Gol', 'Premiu', 'Desconocido', None, 'God', 'Gold', 'silver'],
    'fecha_registro': ['2024-01-10', '2024/02/15', '2024/02/15', '10-03-2024',
                       '2024-04-01', '2024-04-05', '2024-04-28', np.nan, '2024-05-10'],
    'score_riesgo': [0.1, 0.3, 0.3, 0.95, -0.2, 1.5, 0.5, 0.5, 0.1]
})

# Vista rápida
df

,id_cliente,nombre,edad,ingresos_mensuales,ciudad,segmento,fecha_registro,score_riesgo
0,1.0,Ana,25.0,1500.0,Madrid,Silv,2024-01-10,0.10
1,2.0,CARLOS,40.0,2500.5,Barcelona,Gol,2024/02/15,0.30
2,2.0,CARLOS,40.0,2500.5,Barcelona,Gol,2024/02/15,0.30
3,3.0,María,NaN,3200.0,Madrid,Premiu,10-03-2024,0.95
4,4.0,NaN,17.0,500.0,MADRID,Desconocido,2024-04-01,-0.20
5,5.0,Juan,80.0,NaN,Valencia,NaN,2024-04-05,1.50
6,6.0,Lucia,35.0,1800.0,Barcelona,God,2024-04-28,0.50
7,6.0,Lucia,35.0,1800.0,Barcelona,Gold,NaN,0.50
8,NaN,Ana,25.0,1500.0,Madrid,silver,2024-05-10,0.10


Problemas intencionales:

- Valores faltantes en nombre, edad, ingresos_mensuales, id_cliente, fecha_registro, segmento

- Duplicados (id_cliente 2 y 6)

- Inconsistencias de texto ("Madrid" vs "MADRID", "Silv" vs "silver")

- Fechas con formato mixto

- Posibles outliers: edad 17 y 80, score_riesgo -0.2 y 1.5

### Identificación de Valores Faltantes

In [32]:
#Contar valores faltantes por columna
df.isna().sum()

id_cliente            1
nombre                1
edad                  1
ingresos_mensuales    1
ciudad                0
segmento              1
fecha_registro        1
score_riesgo          0
dtype: int64

In [33]:
(df.isna().mean()*100).round(2)



id_cliente            11.11
nombre                11.11
edad                  11.11
ingresos_mensuales    11.11
ciudad                 0.00
segmento              11.11
fecha_registro        11.11
score_riesgo           0.00
dtype: float64

### Identificación de Duplicados

In [34]:
# Detectar filas completamente duplicadas
df[df.duplicated(keep=False)]


,id_cliente,nombre,edad,ingresos_mensuales,ciudad,segmento,fecha_registro,score_riesgo
1,2.0,CARLOS,40.0,2500.5,Barcelona,Gol,2024/02/15,0.3
2,2.0,CARLOS,40.0,2500.5,Barcelona,Gol,2024/02/15,0.3


In [35]:
df[df.duplicated(keep=False, subset=['id_cliente'])]


,id_cliente,nombre,edad,ingresos_mensuales,ciudad,segmento,fecha_registro,score_riesgo
1,2.0,CARLOS,40.0,2500.5,Barcelona,Gol,2024/02/15,0.3
2,2.0,CARLOS,40.0,2500.5,Barcelona,Gol,2024/02/15,0.3
6,6.0,Lucia,35.0,1800.0,Barcelona,God,2024-04-28,0.5
7,6.0,Lucia,35.0,1800.0,Barcelona,Gold,NaN,0.5


In [36]:
# Contar registros por ID Cliente
df['id_cliente'].value_counts()


id_cliente
2.0    2
6.0    2
1.0    1
3.0    1
4.0    1
5.0    1
Name: count, dtype: int64

### Detección de Inconsistencias de Formato

In [37]:
df.dtypes

id_cliente            float64
nombre                    str
edad                  float64
ingresos_mensuales    float64
ciudad                    str
segmento                  str
fecha_registro            str
score_riesgo          float64
dtype: object

In [38]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_cliente          8 non-null      float64
 1   nombre              8 non-null      str    
 2   edad                8 non-null      float64
 3   ingresos_mensuales  8 non-null      float64
 4   ciudad              9 non-null      str    
 5   segmento            8 non-null      str    
 6   fecha_registro      8 non-null      str    
 7   score_riesgo        9 non-null      float64
dtypes: float64(4), str(4)
memory usage: 708.0 bytes


In [39]:
# Analizar inconsistencias en las columnas categoricas
for col in df.select_dtypes(include='str').columns.sort_values():
    print(f'{col}: {list(df[col].unique())}')
    print(f'-' * 150)



ciudad: ['Madrid', 'Barcelona', 'MADRID', 'Valencia']
------------------------------------------------------------------------------------------------------------------------------------------------------
fecha_registro: ['2024-01-10', '2024/02/15', '10-03-2024', '2024-04-01', '2024-04-05', '2024-04-28', nan, '2024-05-10']
------------------------------------------------------------------------------------------------------------------------------------------------------
nombre: [' Ana  ', 'CARLOS', 'María', nan, 'Juan', 'Lucia', 'Ana']
------------------------------------------------------------------------------------------------------------------------------------------------------
segmento: ['Silv', 'Gol', 'Premiu', 'Desconocido', nan, 'God', 'Gold', 'silver']
------------------------------------------------------------------------------------------------------------------------------------------------------


### Detección de Valores Atípicos (outliers) y Fuera de Rango

In [40]:
# Usar describe para ver posibles valores extremos
df.describe()



,id_cliente,edad,ingresos_mensuales,score_riesgo
count,8.00000,8.000000,8.0000,9.000000
mean,3.62500,37.125000,1912.6250,0.450000
std,1.92261,19.134393,820.2722,0.508675
min,1.00000,17.000000,500.0000,-0.200000
25%,2.00000,25.000000,1500.0000,0.100000
50%,3.50000,35.000000,1800.0000,0.300000
75%,5.25000,40.000000,2500.5000,0.500000
max,6.00000,80.000000,3200.0000,1.500000


In [41]:
q1 = df['edad'].quantile(0.25)
q3 = df['edad'].quantile(0.75)

iqr = q3 - q1
lim_inf = q1 - 1.5 * iqr
lim_sup = q3 + 1.5 * iqr
outliers = df[(df['edad'] < lim_inf) | (df['edad'] > lim_sup)]
print(f'limites teóricos: {lim_inf}, {lim_sup}')
outliers

limites teóricos: 2.5, 62.5


,id_cliente,nombre,edad,ingresos_mensuales,ciudad,segmento,fecha_registro,score_riesgo
5,5.0,Juan,80.0,NaN,Valencia,NaN,2024-04-05,1.5


In [42]:
outliers_score = df[(df['score_riesgo'] < 0) | (df['score_riesgo'] > 1)]
outliers_score

,id_cliente,nombre,edad,ingresos_mensuales,ciudad,segmento,fecha_registro,score_riesgo
4,4.0,NaN,17.0,500.0,MADRID,Desconocido,2024-04-01,-0.2
5,5.0,Juan,80.0,NaN,Valencia,NaN,2024-04-05,1.5


## U3.02 – Tratamiento de datos faltantes.

Estrategias: eliminar vs imputar

- Eliminar filas/columnas cuando:

    - El porcentaje de nulos es muy alto

    - La fila está casi vacía

- Imputar (rellenar) cuando:

    - La columna es importante

    - Se puede estimar un valor razonable (media, mediana, moda, reglas de negocio)


### Eliminación de filas completas

In [43]:
df


,id_cliente,nombre,edad,ingresos_mensuales,ciudad,segmento,fecha_registro,score_riesgo
0,1.0,Ana,25.0,1500.0,Madrid,Silv,2024-01-10,0.10
1,2.0,CARLOS,40.0,2500.5,Barcelona,Gol,2024/02/15,0.30
2,2.0,CARLOS,40.0,2500.5,Barcelona,Gol,2024/02/15,0.30
3,3.0,María,NaN,3200.0,Madrid,Premiu,10-03-2024,0.95
4,4.0,NaN,17.0,500.0,MADRID,Desconocido,2024-04-01,-0.20
5,5.0,Juan,80.0,NaN,Valencia,NaN,2024-04-05,1.50
6,6.0,Lucia,35.0,1800.0,Barcelona,God,2024-04-28,0.50
7,6.0,Lucia,35.0,1800.0,Barcelona,Gold,NaN,0.50
8,NaN,Ana,25.0,1500.0,Madrid,silver,2024-05-10,0.10


In [44]:
df_sin_nulos = df.dropna()
df_sin_nulos

,id_cliente,nombre,edad,ingresos_mensuales,ciudad,segmento,fecha_registro,score_riesgo
0,1.0,Ana,25.0,1500.0,Madrid,Silv,2024-01-10,0.1
1,2.0,CARLOS,40.0,2500.5,Barcelona,Gol,2024/02/15,0.3
2,2.0,CARLOS,40.0,2500.5,Barcelona,Gol,2024/02/15,0.3
6,6.0,Lucia,35.0,1800.0,Barcelona,God,2024-04-28,0.5


In [47]:
df_id_nulos = df.dropna(subset=['id_cliente'])
df_id_nulos

,id_cliente,nombre,edad,ingresos_mensuales,ciudad,segmento,fecha_registro,score_riesgo
0,1.0,Ana,25.0,1500.0,Madrid,Silv,2024-01-10,0.10
1,2.0,CARLOS,40.0,2500.5,Barcelona,Gol,2024/02/15,0.30
2,2.0,CARLOS,40.0,2500.5,Barcelona,Gol,2024/02/15,0.30
3,3.0,María,NaN,3200.0,Madrid,Premiu,10-03-2024,0.95
4,4.0,NaN,17.0,500.0,MADRID,Desconocido,2024-04-01,-0.20
5,5.0,Juan,80.0,NaN,Valencia,NaN,2024-04-05,1.50
6,6.0,Lucia,35.0,1800.0,Barcelona,God,2024-04-28,0.50
7,6.0,Lucia,35.0,1800.0,Barcelona,Gold,NaN,0.50


### Imputación de datos.

In [50]:
# Calcular media y mediana de edad

promedio_edad = df['edad'].mean()
mediana_edad = df['edad'].median()

df['edad_fill'] = df['edad'].fillna(mediana_edad)
df[['edad', 'edad_fill']]


,edad,edad_fill
0,25.0,25.0
1,40.0,40.0
2,40.0,40.0
3,NaN,35.0
4,17.0,17.0
5,80.0,80.0
6,35.0,35.0
7,35.0,35.0
8,25.0,25.0


In [54]:
promedio_edad = df['edad'].mean()
mediana_edad = df['edad'].median()

# Imputación con mediana
df['edad_fill'] = df['edad'].fillna(mediana_edad)

#Forward fill: Propagar último valor hacia adelante
df['fecha_registro_ffill'] = df['fecha_registro'].ffill()

df[['edad', 'edad_fill', 'fecha_registro', 'fecha_registro_ffill']]


,edad,edad_fill,fecha_registro,fecha_registro_ffill
0,25.0,25.0,2024-01-10,2024-01-10
1,40.0,40.0,2024/02/15,2024/02/15
2,40.0,40.0,2024/02/15,2024/02/15
3,NaN,35.0,10-03-2024,10-03-2024
4,17.0,17.0,2024-04-01,2024-04-01
5,80.0,80.0,2024-04-05,2024-04-05
6,35.0,35.0,2024-04-28,2024-04-28
7,35.0,35.0,NaN,2024-04-28
8,25.0,25.0,2024-05-10,2024-05-10


## U3.03 – Normalización de Texto y Categorías

### Limpiar y normalizar texto

### Normalización de Categorías

### Detección y Limpieza de Patrones Específicos

## 3.04 Transformación y Estandarización de Fechas

### Conversión de Formatos de Fecha

### Extracción de componentes de fecha

### Conversión de Tipos de Datos

## U3.05 – Consolidación Final - Tratamiento de Duplicados y Validación

### Eliminación de Duplicados Exactos

### Eliminación de Duplicados por columnas

### Validación básica de calidad

### Exportar Dataset Final Limpio